# Classic Ranking Models on Ali-CCP (Colab)

Compares three foundational CTR/CTCVR ranking architectures on the full Ali-CCP dataset:

| Model | Paper | Key idea |
|-------|-------|----------|
| **Wide & Deep** | Cheng et al., Google KDD 2016 | Memorization (wide linear) + Generalization (deep MLP) |
| **DeepFM** | Guo et al., HuaWei IJCAI 2017 | FM implicit pairwise interactions + deep MLP |
| **DCN V2** | Wang et al., Google WWW 2021 | Explicit polynomial cross features (parallel structure) |

All models output `(p_ctr, p_cvr, p_ctcvr)` and use the ESMM entire-space training paradigm
for a fair apples-to-apples comparison with the existing ESMM/MMoE/PLE results.

**Data:** Ali-CCP full split (42M train / 43M test rows). Reuses preprocessed Parquet from prior runs.


In [ ]:
import os
if os.path.ismount('/content/drive'):
    print('Drive already mounted.')
else:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception:
        print('Skipping drive mount (not in Colab UI or drive unavailable).')

In [ ]:
import os
WORK_DIR = '/content/drive/MyDrive/colab/recsys_playground'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

In [ ]:
import os, subprocess, shutil
try:
    import google.colab
    IN_COLAB = True
except Exception:
    IN_COLAB = False

repo_url = 'https://github.com/allyoushawn/recsys_playground.git'
repo_dir = 'recsys_playground'
branch_name = 'main'

SKIP_GIT_REPO_SYNC = True
if os.environ.get('FORCE_GIT_SYNC', '').strip().lower() in ('1', 'true', 'yes'):
    SKIP_GIT_REPO_SYNC = False

if IN_COLAB:
    git_marker = os.path.join(repo_dir, '.git')
    if SKIP_GIT_REPO_SYNC:
        if os.path.isdir(repo_dir):
            os.chdir(repo_dir)
            print('[SKIP_GIT_REPO_SYNC] Using Drive copy.')
        else:
            subprocess.run(['git', 'clone', repo_url], check=True)
            os.chdir(repo_dir)
            subprocess.run(['git', 'checkout', branch_name], check=False)
    elif os.path.isdir(repo_dir) and os.path.isdir(git_marker):
        os.chdir(repo_dir)
        subprocess.run(['git', 'fetch', '--all'], check=True)
        subprocess.run(['git', 'checkout', branch_name], check=False)
        subprocess.run(['git', 'reset', '--hard', f'origin/{branch_name}'], check=False)
    else:
        if os.path.exists(repo_dir):
            shutil.rmtree(repo_dir)
        subprocess.run(['git', 'clone', repo_url], check=True)
        os.chdir(repo_dir)
        subprocess.run(['git', 'fetch', '--all'], check=True)
        subprocess.run(['git', 'checkout', branch_name], check=False)

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'torch', 'pandas', 'numpy', 'scikit-learn',
                'matplotlib', 'seaborn', 'requests', 'tqdm', 'joblib',
                'pyarrow', 'psutil'], check=True)

In [ ]:
import os

PROJECT_NAME = 'classic_models_experiment'
DATA_DIR = '/content/drive/MyDrive/colab/data/ali_ccp'
# Reuse preprocessed Parquet from prior ESMM runs (same pipeline)
PROCESSED_FULL_DIR = os.path.join(DATA_DIR, 'processed_esmm_full_parquet')
ROUND_RESULTS_DIR  = os.path.join(DATA_DIR, 'classic_models_cache')
for _d in (PROCESSED_FULL_DIR, ROUND_RESULTS_DIR):
    os.makedirs(_d, exist_ok=True)

PREPROCESSED_TRAIN          = os.path.join(PROCESSED_FULL_DIR, 'preprocessed_train.parquet')
PREPROCESSED_TEST           = os.path.join(PROCESSED_FULL_DIR, 'preprocessed_test.parquet')
PREPROCESSED_SPARSE_VOCAB_CACHE = os.path.join(PROCESSED_FULL_DIR, 'preprocessed_sparse_vocab.pkl')

# Local SSD paths — test Parquet is copied here before eval to avoid Drive I/O bottleneck
LOCAL_CONTENT_DIR   = '/content'
LOCAL_TEST_PARQUET  = os.path.join(LOCAL_CONTENT_DIR, 'preprocessed_test_local.parquet')

WIDE_DEEP_RESULTS_JSON = os.path.join(ROUND_RESULTS_DIR, 'wide_deep_results.json')
DEEPFM_RESULTS_JSON    = os.path.join(ROUND_RESULTS_DIR, 'deepfm_results.json')
DCNV2_RESULTS_JSON     = os.path.join(ROUND_RESULTS_DIR, 'dcnv2_results.json')

# Empty = reuse all cached result JSONs; add names ('wide_deep','deepfm','dcnv2') to force retrain
CLEAN_EXPERIMENT_JSON = []

CLEAN_PREPROCESSED_PARQUET     = False  # Reuse from ESMM runs
CLEAN_PREPROCESSED_VOCAB_CACHE = False
FORCE_REBUILD_PREPROCESSED_VOCAB = False

def _drive_remove(path, desc):
    if os.path.isfile(path):
        os.remove(path)
        print(f'[cleanup] Removed {desc}: {path}')

_CLEAN_MAP = {
    'wide_deep': WIDE_DEEP_RESULTS_JSON,
    'deepfm':    DEEPFM_RESULTS_JSON,
    'dcnv2':     DCNV2_RESULTS_JSON,
}
for _k in CLEAN_EXPERIMENT_JSON:
    if _k not in _CLEAN_MAP:
        raise ValueError(f'Unknown key: {_k!r}')
    _drive_remove(_CLEAN_MAP[_k], f'{_k}_results.json')

# --- What to run ---
RUN_WIDE_DEEP = True
RUN_DEEPFM    = True
RUN_DCNV2     = True
_RUN_ANY = RUN_WIDE_DEEP or RUN_DEEPFM or RUN_DCNV2

# --- I/O sizes (same as ESMM notebook) ---
STREAM_PARSE_CHUNK_ROWS  = 500_000
VOCAB_SCAN_ROWS_PER_BATCH = 200_000
NORM_STREAM_BATCH_ROWS   = 500_000
TRAIN_BATCH_SIZE = 4096
EVAL_BATCH_SIZE  = 500_000

# --- Training opts ---
TRAIN_USE_AMP            = True
TRAIN_PREFETCH_ROW_GROUPS = True
TRAIN_USE_MANUAL_BATCHES = True
TRAIN_READ_ROW_GROUPS_AS_ARROW = False
TRAIN_USE_TORCH_COMPILE  = False

RANDOM_STATE = 42
EMBED_DIM    = 18

SPARSE_COLS = ['101','121','122','124','125','126','127','128','129',
               '205','206','207','210','216','508','509','702','853',
               '301','109_14','110_14','127_14','150_14']
DENSE_COLS  = ['109_14','110_14','127_14','150_14','508','509','702','853']
DENSE_FEAT_COLS = ['D' + c for c in DENSE_COLS]

print('Config:')
print(f'  RUN_WIDE_DEEP={RUN_WIDE_DEEP}  RUN_DEEPFM={RUN_DEEPFM}  RUN_DCNV2={RUN_DCNV2}')
print(f'  EMBED_DIM={EMBED_DIM}, results under {ROUND_RESULTS_DIR}/')
print(f'  CLEAN_EXPERIMENT_JSON={CLEAN_EXPERIMENT_JSON}')

In [ ]:
import os, sys
from pathlib import Path

_ESMM_DIR = (Path.cwd() / 'experiments' / '20260404_ali_cpp_esmm').resolve()
if _ESMM_DIR.is_dir() and str(_ESMM_DIR) not in sys.path:
    sys.path.insert(0, str(_ESMM_DIR))

_NEW_MODELS_DIR = (Path.cwd() / 'experiments' / '20260519_wide_deep_deepfm_dcn').resolve()
if _NEW_MODELS_DIR.is_dir() and str(_NEW_MODELS_DIR) not in sys.path:
    sys.path.insert(0, str(_NEW_MODELS_DIR))

from esmm_ali_ccp_impl import (
    COMMON_FEATURES_TRAIN, COMMON_FEATURES_TEST,
    SAMPLE_SKELETON_TRAIN, SAMPLE_SKELETON_TEST,
    SAMPLE_TRAIN_TAR, SAMPLE_TEST_TAR,
    TRAIN_CSV, TEST_CSV,
    find_file_recursive,
)
import tarfile
os.makedirs(DATA_DIR, exist_ok=True)

train_tar_path = os.path.join(DATA_DIR, SAMPLE_TRAIN_TAR)
test_tar_path  = os.path.join(DATA_DIR, SAMPLE_TEST_TAR)
has_archives   = os.path.isfile(train_tar_path) and os.path.isfile(test_tar_path)
needs_extract  = has_archives and (
    not find_file_recursive(DATA_DIR, SAMPLE_SKELETON_TRAIN) or
    not find_file_recursive(DATA_DIR, COMMON_FEATURES_TRAIN))
if needs_extract:
    for arc in [SAMPLE_TRAIN_TAR, SAMPLE_TEST_TAR]:
        p = os.path.join(DATA_DIR, arc)
        if os.path.isfile(p):
            print(f'Extracting {arc}...')
            with tarfile.open(p, 'r:*') as tf:
                tf.extractall(DATA_DIR)

has_raw    = (find_file_recursive(DATA_DIR, SAMPLE_SKELETON_TRAIN) and
              find_file_recursive(DATA_DIR, COMMON_FEATURES_TRAIN))
has_splits = (os.path.isfile(os.path.join(DATA_DIR, TRAIN_CSV)) and
              os.path.isfile(os.path.join(DATA_DIR, TEST_CSV)))

if _RUN_ANY and not (has_raw or has_splits):
    raise FileNotFoundError(
        f'No Ali-CCP data at {DATA_DIR}. Need raw tars or splits.')
print('Raw data OK.')

In [ ]:
from esmm_ali_ccp_impl import *
import esmm_ali_ccp_impl as _impl

_impl.DEFAULT_STREAM_PARSE_CHUNK_ROWS    = STREAM_PARSE_CHUNK_ROWS
_impl.DEFAULT_VOCAB_SCAN_ROWS_PER_BATCH = VOCAB_SCAN_ROWS_PER_BATCH
_impl.DEFAULT_NORM_STREAM_BATCH_ROWS    = NORM_STREAM_BATCH_ROWS
_impl.DEFAULT_EVAL_TEST_BATCH_ROWS      = EVAL_BATCH_SIZE

# Import new model classes
from new_models_impl import WideAndDeepModel, DeepFMModel, DCNv2Model
print('Models loaded: WideAndDeepModel, DeepFMModel, DCNv2Model')

In [ ]:
import gc, time, psutil, torch

if not _RUN_ANY:
    print('All flags False — skipping data prep.')
else:
    # Use ensure_full_split_parquet_streaming to get parsed parquet path.
    # This is required for vocab cache compatibility: the PKL was built from
    # the parsed (pre-normalization) parquet path, so we must pass the same path.
    t0 = time.time()
    p_train, p_test = ensure_full_split_parquet_streaming(
        DATA_DIR, PROCESSED_FULL_DIR, SPARSE_COLS, DENSE_COLS, DENSE_FEAT_COLS)
    print(f'Parsed parquet paths ready in {time.time() - t0:.1f}s')

    t1 = time.time()
    print('Vocab (cache or scan)...')
    vocabs, sparse_cardinalities = load_or_build_sparse_vocabs_filtered_parquet(
        p_train, SPARSE_COLS, min_count=5,
        cache_path=PREPROCESSED_SPARSE_VOCAB_CACHE,
        force_rebuild=FORCE_REBUILD_PREPROCESSED_VOCAB,
        vocab_scan_rows_per_batch=VOCAB_SCAN_ROWS_PER_BATCH)
    print(f'Vocab ready in {time.time() - t1:.1f}s')

    # Copy test Parquet to local SSD to avoid Drive I/O bottleneck during full eval
    if not os.path.exists(LOCAL_TEST_PARQUET):
        import shutil
        print(f'[local SSD] Copying test Parquet to {LOCAL_TEST_PARQUET} ...')
        t_copy = time.time()
        shutil.copy2(p_test, LOCAL_TEST_PARQUET)
        copy_mb = os.path.getsize(LOCAL_TEST_PARQUET) / 1e6
        print(f'[local SSD] Copied {copy_mb:.0f} MB in {time.time()-t_copy:.1f}s')
    else:
        print(f'[local SSD] Test Parquet already cached at {LOCAL_TEST_PARQUET}')

    if not (os.path.isfile(PREPROCESSED_TRAIN) and os.path.isfile(PREPROCESSED_TEST)):
        print('Streaming dense log1p -> normalized Parquet...')
        t2 = time.time()
        stream_normalize_parquet(
            p_train, PREPROCESSED_TRAIN, SPARSE_COLS, DENSE_FEAT_COLS,
            norm_stream_batch_rows=NORM_STREAM_BATCH_ROWS)
        stream_normalize_parquet(
            p_test, PREPROCESSED_TEST, SPARSE_COLS, DENSE_FEAT_COLS,
            norm_stream_batch_rows=NORM_STREAM_BATCH_ROWS)
        print(f'Normalize done in {time.time() - t2:.1f}s')
    else:
        print(f'Reusing {PREPROCESSED_TRAIN} (set CLEAN_PREPROCESSED_PARQUET=True to rebuild)')

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f'RAM: {psutil.Process().memory_info().rss / 1024**3:.1f} GB')

In [ ]:
import json, time, gc, numpy as np, torch

def _eval_full(model, test_path, vocabs, sparse_cols, dense_feat_cols, max_eval_rows=None, batch_rows=500_000, inner_batch=8192):
    """Eval on up to max_eval_rows (None = all rows). Local SSD eliminates Drive I/O; CPU encoding is the bottleneck."""
    import pyarrow.parquet as pq
    from torch.utils.data import DataLoader, TensorDataset
    from sklearn.metrics import roc_auc_score
    enc_tables = {col: (tuple(vocabs[col].keys()), np.array(list(vocabs[col].values()), dtype=np.int32)) for col in sparse_cols}
    cols = sparse_cols + dense_feat_cols + ['click', 'purchase']
    # GPU keepalive: matmul every 50ms keeps utilization visible to Colab's idle detector
    import threading as _threading
    _keepalive_active = [True]
    def _keepalive_fn():
        if torch.cuda.is_available():
            _m = torch.randn(512, 512, device='cuda')
            while _keepalive_active[0]:
                torch.mm(_m, _m)
                _threading.Event().wait(0.05)
    t_keepalive = _threading.Thread(target=_keepalive_fn, daemon=True)
    t_keepalive.start()
    model.eval()
    ctr_p, ctr_y, ctcvr_p, ctcvr_y, cvr_p, cvr_y = [], [], [], [], [], []
    rows_seen = 0
    pf = pq.ParquetFile(test_path)
    _iter_batch_size = batch_rows if max_eval_rows is None else min(batch_rows, max_eval_rows)
    for batch in pf.iter_batches(batch_size=_iter_batch_size, columns=cols):
        if max_eval_rows is not None and rows_seen >= max_eval_rows:
            break
        df = batch.to_pandas()
        rows_seen += len(df)
        sp, dn, _ = encode_and_tensorize_fast(df, enc_tables, sparse_cols, dense_feat_cols, 'purchase')
        y_click = torch.from_numpy(df['click'].values.astype(np.float32))
        y_pur   = torch.from_numpy(df['purchase'].values.astype(np.float32))
        y_ctcvr = y_click * y_pur
        del df
        loader = DataLoader(TensorDataset(sp, dn, y_click, y_pur, y_ctcvr), batch_size=inner_batch, shuffle=False)
        with torch.no_grad():
            for spb, dnb, ycb, ypb, yccb in loader:
                spb, dnb = spb.to(device), dnb.to(device)
                pc, pv, pcc = model(spb, dnb)
                ctr_p.append(pc.cpu().numpy()); ctr_y.append(ycb.numpy())
                ctcvr_p.append(pcc.cpu().numpy()); ctcvr_y.append(yccb.numpy())
                m = ycb.numpy() > 0.5
                if m.any():
                    cvr_p.append(pv.cpu().numpy()[m]); cvr_y.append(ypb.numpy()[m])
        del sp, dn, y_click, y_pur, y_ctcvr
    ctr_p = np.concatenate(ctr_p); ctr_y = np.concatenate(ctr_y)
    ctcvr_p = np.concatenate(ctcvr_p); ctcvr_y = np.concatenate(ctcvr_y)
    cvr_p = np.concatenate(cvr_p) if cvr_p else np.array([], dtype=np.float32)
    cvr_y = np.concatenate(cvr_y) if cvr_y else np.array([], dtype=np.float32)
    out = {'eval_rows': rows_seen}
    if len(np.unique(ctr_y)) >= 2:
        out['CTR_AUC'] = float(roc_auc_score(ctr_y, ctr_p))
    if len(np.unique(ctcvr_y)) >= 2:
        out['CTCVR_AUC'] = float(roc_auc_score(ctcvr_y, ctcvr_p))
    if len(np.unique(cvr_y)) >= 2:
        out['CVR_AUC'] = float(roc_auc_score(cvr_y, cvr_p))
    _keepalive_active[0] = False
    return out

if not _RUN_ANY:
    print('All flags False — skipping training.')
else:
    _TRAIN_KW = dict(
        epochs=1,  # 1 epoch = ~7min; session-safe
        batch_size=TRAIN_BATCH_SIZE,
        lr=1e-3,
        seed=RANDOM_STATE,
        embed_dim=EMBED_DIM,
        max_wall_seconds=None,
        max_optimizer_steps=None,
        max_batches_per_epoch=None,
        max_row_groups_per_epoch=None,  # full data
        use_amp=TRAIN_USE_AMP,
        prefetch_row_groups=TRAIN_PREFETCH_ROW_GROUPS,
        use_manual_batches=TRAIN_USE_MANUAL_BATCHES,
        read_row_groups_as_arrow=TRAIN_READ_ROW_GROUPS_AS_ARROW,
        use_torch_compile=TRAIN_USE_TORCH_COMPILE,
    )

    def _jsonify(o):
        if isinstance(o, dict):   return {k: _jsonify(v) for k, v in o.items()}
        if isinstance(o, (list, tuple)): return [_jsonify(v) for v in o]
        if isinstance(o, (np.floating, np.integer)): return o.item()
        return o

    def _fmt(x):
        return 'nan' if x is None or (isinstance(x, float) and x != x) else f'{float(x):.4f}'

    def _run_model(name, run_flag, result_path, ctor, ctor_kw, desc):
        if not run_flag:
            print(f'{name}: skipped (flag False)')
            return
        if os.path.isfile(result_path):
            with open(result_path) as f:
                r = json.load(f)
            print(f'{name}: cached  CTCVR={_fmt(r.get("CTCVR_AUC"))}  CTR={_fmt(r.get("CTR_AUC"))}  CVR={_fmt(r.get("CVR_AUC"))}')
            return
        print('\n' + '='*60)
        print(f'{name} — {desc}')
        print('='*60)
        t0 = time.time()
        model, _, meta = train_esmm_parquet_rowgroups(
            PREPROCESSED_TRAIN, vocabs, sparse_cardinalities,
            SPARSE_COLS, DENSE_FEAT_COLS,
            model_ctor=ctor, model_ctor_kwargs=ctor_kw,
            **_TRAIN_KW,
        )
        wall = time.time() - t0
        n_params = int(sum(p.numel() for p in model.parameters()))
        metrics = _eval_full(model, LOCAL_TEST_PARQUET, vocabs, SPARSE_COLS, DENSE_FEAT_COLS)
        for mk, mv in metrics.items():
            print(f'  {mk}: {_fmt(mv)}')
        out = {**metrics, 'wall_clock_seconds': int(wall), 'num_parameters': n_params,
               'train_samples_per_sec': float(meta['samples_per_sec']),
               'train_wall_seconds': float(meta['train_wall_seconds']),
               'early_stop_reason': meta['early_stop_reason'],
               'max_row_groups_per_epoch': None}
        with open(result_path, 'w') as f:
            json.dump(_jsonify(out), f)
        del model; gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # --- Run models ---
    _run_model('Wide & Deep', RUN_WIDE_DEEP, WIDE_DEEP_RESULTS_JSON,
               WideAndDeepModel, {'deep_dims': (360, 200, 80)},
               'Wide (linear) + Deep (MLP), Cheng et al. 2016')

    _run_model('DeepFM', RUN_DEEPFM, DEEPFM_RESULTS_JSON,
               DeepFMModel, {'dnn_dims': (360, 200, 80)},
               'FM pairwise interactions + DNN, Guo et al. 2017')

    _run_model('DCN V2', RUN_DCNV2, DCNV2_RESULTS_JSON,
               DCNv2Model, {'num_cross_layers': 3, 'deep_dims': (360, 200, 80)},
               'Parallel cross network + DNN, Wang et al. 2021')

    # --- Summary ---
    print('\n' + '='*60)
    print('SUMMARY — Classic Ranking Models on Ali-CCP (full data)')
    print('='*60)
    _models = [
        ('Wide & Deep', WIDE_DEEP_RESULTS_JSON),
        ('DeepFM',      DEEPFM_RESULTS_JSON),
        ('DCN V2',      DCNV2_RESULTS_JSON),
    ]
    for label, path in _models:
        if not os.path.isfile(path):
            print(f'  {label}: (no cache)')
            continue
        with open(path) as f:
            r = json.load(f)
        print(f'  {label:12s}  CTCVR={_fmt(r.get("CTCVR_AUC"))}  '
              f'CTR={_fmt(r.get("CTR_AUC"))}  CVR={_fmt(r.get("CVR_AUC"))}  '
              f'params={r.get("num_parameters",0):,}  wall={r.get("wall_clock_seconds",0)}s')

    print('\n  Reference (from prior ESMM runs):')
    print('  ESMM_K_ref   CTCVR=0.6114')
    print('  ESMM_MMoE    CTCVR=0.6164  (prior best)')